# JSON Files Ingestion
1. Read active sources from the **control table**
2. List the **dated folders** (e.g. `20260621`) inside each source path
3. Skip folders already loaded into the target table, process only the **new** ones

In [0]:
from pyspark.sql import functions as F

## 1. Config

In [0]:
catalog = "iics"
schema = "casemix"
source_control_table = "json_control_test"
max_files_per_trigger = 50000

In [0]:
%sql
drop table if exists iics.casemix.json_control_test

In [0]:
%sql
-- drop table if exists iics.casemix.json_test_v2

## 2. Control table (created once, never dropped)

In [0]:
ctl = f"{catalog}.{schema}.{source_control_table}"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ctl} (
        s_no INT, source STRING, file_path STRING, file_type STRING,
        tgt_catalog STRING, tgt_schema STRING, tgt_table STRING, is_active INT
    ) USING DELTA
""")

if spark.table(ctl).count() == 0:
    spark.sql(f"""
        INSERT INTO {ctl} VALUES
        -- (1, 's3', '/Volumes/iics/casemix/history/casemix/zip_files/processed/', 'json',
        --  'iics', 'casemix', 'json_test_v2', 1)
        (2, 's3', '/Volumes/iics/casemix/history/json_stress_test/', 'json',
         'iics', 'casemix', 'json_test_v3', 1)
    """)

display(spark.table(ctl))

## 3. Process new dated folders

In [0]:
configs = spark.table(ctl).filter("is_active = 1").orderBy("s_no").collect()
for src in configs:

    src_path = src.file_path.rstrip("/") + "/"
    target = f"{src.tgt_catalog}.{src.tgt_schema}.{src.tgt_table}"

    # dated folders in the source path (8-digit names only)
    all_folders = sorted(
        f.name.rstrip("/") for f in dbutils.fs.ls(src_path)
        if f.name.endswith("/") and f.name.rstrip("/").isdigit() and len(f.name.rstrip("/")) == 8
    )

    # folders already loaded
    done = set()
    if spark.catalog.tableExists(target):
        done = {r.source_folder for r in spark.table(target).select("source_folder").distinct().collect()}

    new_folders = [f for f in all_folders if f not in done]
    print(f"{src_path}: {len(all_folders)} folders, {len(new_folders)} new")

    for dated_folder in new_folders:
        check_point_path = f"{src_path}checkpoint/{dated_folder}"

        m = F.col("_metadata")

        raw = (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "text")
            .option("wholetext", "true")
            .option("cloudFiles.maxFilesPerTrigger", max_files_per_trigger)
            .load(f"{src_path}{dated_folder}/")
        )
        print("-----------------m-------------------")
        print(m)
        print("------------m[file_name]-------------")
        print(f"{m["file_name"]}")
        name_dt = F.regexp_extract(m["file_name"], r"_(\d{8})_\d{6}\.[A-Za-z]+$", 1)
        print("---------------name_dt---------------")
        print(name_dt)
        print("------------dt_file-------------------")
        dt_file = F.when(name_dt != "", name_dt).otherwise(None)
        print(dt_file)

        df = (
            raw.select(
                F.lit(f"{src_path}{dated_folder}/").alias("file_path"),
                F.lit(dated_folder).alias("source_folder"),
                m["file_name"].alias("file_name"),
                F.col("value").alias("json_original_data"),
                F.expr("try_parse_json(value)").alias("json_data_for_parsing"),
                (m["file_size"] / 1024.0).alias("file_size_kb"),
                m["file_modification_time"].alias("file_modified_date"),
                dt_file.alias("dt_data_file"),
            )
            .withColumn("dt_data_file_date", F.to_date("dt_data_file", "yyyyMMdd"))
            .withColumn("data_year", F.year("dt_data_file_date"))
            .withColumn("data_month", F.month("dt_data_file_date"))
            .withColumn("loaded_at", F.current_timestamp())
        )

        q = (
            df.writeStream.format("delta")
            .option("checkpointLocation", check_point_path)
            .trigger(availableNow=True)
            .partitionBy("data_year", "data_month")
            .toTable(target)
        )
        q.awaitTermination()
        print(f"  loaded {dated_folder}")

## 4. Check

In [0]:
for src in spark.table(ctl).filter("is_active = 1").collect():
    target = f"{src.tgt_catalog}.{src.tgt_schema}.{src.tgt_table}"
    display(spark.table(target).groupBy("source_folder").count().orderBy("source_folder"))

In [0]:
%sql
SELECT file_name,dt_data_file,dt_data_file_date,data_year,data_month FROM iics.casemix.json_test_v2

In [0]:
%sql
SELECT * FROM iics.casemix.json_test_v2 WHERE dt_data_file_date is not null

In [0]:
%sql
SELECT * FROM iics.casemix.json_stress_test_v2 LIMIT 10

In [0]:
%sql
SELECT COUNT(*) FROM iics.casemix.json_stress_test_v2 LIMIT 10